# 1 Gate, highway, or route: does a mechanism earn its weights?

At the end of this notebook you can do three things:

- **Point at the control of an architecture claim.** When a model with a new part beats a
  smaller model, you can say why that proves little (the bigger model may win only because it
  is bigger), and name the model that does prove it: the same model without the new part, with
  as many weights, trained on the same data in the same way.
- **Draw a highway and a mixture of experts as routes.** You can say that a gate is one part
  of a parallel architecture: a highway weighs a block with `g` and the skip around it with
  `1 − g`; a mixture of experts has more routes and a router that gives every expert a weight.
  You can point at the routes that are added and at the shapes that must match.
- **Tell the weights a sparse mixture of experts trains from the weights it uses.** You can
  count both, say why using fewer weights per input does not make a model faster by itself,
  recognise router collapse (all inputs sent to one expert) and say what load balancing does
  about it.

The exercise of this lesson (`05_exercise`) asks you to build a routed architecture on the
heartbeats and to put it against a control with as many weights. This notebook does the same
on a small forecasting problem where a training run takes seconds, so every model can be
trained five times and every claim can be checked against its spread.

## 1.0 The data and the task

The series is the electricity demand of one Australian state (series `T1` of the
`australian_electricity_demand` set in the Monash forecasting archive), in megawatts (MW),
measured every half hour from 2002 to early 2015. Demand follows people: low at night, high
in the day, lower at weekends, higher in a hot summer or a cold winter.

**The task.** From four days of demand (192 half-hours) predict the next day (48 half-hours).
One example is a **window** of 240 consecutive half-hours: the first 192 are the input, the
last 48 the target. A model is a function from a `(B, 192)` batch to a `(B, 48)` batch.

**The split in time.** The last full year is the **test** split, the year before it the
**validation** split (`val`), everything earlier **train**. A split in time, not at random:
a forecast is used on days that come after the data it learned from, and a random split
would put windows from the same week in train and in val. As in lesson 2: train fits the
weights, val chooses (here: the epoch to keep, and which model is better), test is scored
once at the end.

**Windows and scale.** Train windows start every 24 half-hours (`STRIDE = 24`), so
neighbouring windows overlap; val and test windows start every 48, one per day. Every value is
scaled with the mean and standard deviation of the train span, so the models see numbers
around 0, and an error of 0.1 is a tenth of that standard deviation. The models train on the
mean squared error (MSE) and are reported in the mean absolute error (MAE) on that scale:
the MSE punishes large misses and gives a smooth gradient, the MAE is the average miss and is
easier to read.

The first cell loads the series, cuts the windows and draws both.

In [ ]:
import logging

# a torch module that vectormesh imports warns on macOS that it cannot redirect output;
# nothing in this notebook uses it
logging.getLogger("torch.distributed.elastic.multiprocessing.redirects").setLevel(
    logging.ERROR
)

In [ ]:
import sys
import time
from dataclasses import dataclass
from functools import partial
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from australian_electricity import load_ausalia
from loguru import logger
from notebooktester import param
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from vectormesh.components import (
    Gate,
    Highway,
    MoE,
    NeuralNet,
    SparseMoE,
    load_balancing_loss,
)
from vectormesh.components.metrics import MAE
from vectormesh.training import Trainer, TrainerSettings
from windowing import window_index

# the Trainer logs every epoch at INFO; keep warnings only
logger.remove()
logger.add(sys.stderr, level="WARNING")

DAY = 48  # half-hourly steps per day
W, H = 4 * DAY, DAY  # four days in, one day out
STRIDE = 24

ausalia = load_ausalia()
print(ausalia)
mu = ausalia.values[ausalia.splits["train"]].mean()
sigma = ausalia.values[ausalia.splits["train"]].std()
print(
    f"train mean {mu:.0f} MW, train std {sigma:.0f} MW: an MAE of 0.1 is {0.1 * sigma:.0f} MW"
)


def windows(split: str, stride: int) -> tuple[torch.Tensor, torch.Tensor]:
    values = (ausalia.values[ausalia.splits[split]] - mu) / sigma
    win = values[window_index(len(values), W + H, stride)]
    return win[:, :W], win[:, W:]


xtr, ytr = windows("train", STRIDE)
xva, yva = windows("val", 2 * STRIDE)
xte, yte = windows("test", 2 * STRIDE)
print(f"train {tuple(xtr.shape)}   val {tuple(xva.shape)}   test {tuple(xte.shape)}")

fig, (top, bottom) = plt.subplots(2, 1, figsize=(11, 6))
daily = ausalia.values.reshape(-1, DAY).mean(1).numpy()
days = ausalia.timestamps[::DAY]
for split, colour in [("train", "C0"), ("val", "C1"), ("test", "C2")]:
    s = ausalia.splits[split]
    part = slice(s.start // DAY, s.stop // DAY)
    top.plot(days[part], daily[part], color=colour, lw=0.6, label=split)
top.set_ylabel("daily mean demand (MW)")
top.set_title(
    "The whole series, one point per day: train, then a year of val, then a year of test"
)
top.legend(fontsize=8, loc="upper left")

first = 10  # one val window, drawn in MW
hours = np.arange(W + H) / 2
xs, ys = xva[first] * sigma + mu, yva[first] * sigma + mu
bottom.plot(hours[:W], xs, color="C1", label="input: 192 half-hours (4 days)")
bottom.plot(hours[W:], ys, color="k", label="target: the next 48 half-hours (1 day)")
bottom.axvline(W / 2, color="grey", ls="--", lw=1)
bottom.set_xlabel("hours from the start of the window")
bottom.set_ylabel("demand (MW)")
bottom.set_title("One val window: the model sees four days and predicts the fifth")
bottom.legend(fontsize=8, loc="upper left")
plt.tight_layout()
plt.show()

The top panel is the whole series, one point per day: demand rises and falls over the
year, with day-to-day noise on top. Val and test are the last two years, after every train
day. The bottom panel is one val window: a daily cycle, low at night and high in the day, in
each of the four input days, and the target day continues it. There are 8,145 train
windows and 361 val and 361 test windows, one per day of each year (the first four days of a
year only serve as input).

A model that has learned the daily shape and the level of the last days already does much of
the job. The question of this notebook is whether a gate or a router adds anything to that.

## 1.1 One gate, three uses: gate, highway, mixture of experts

Lesson 2 (2.2.3) built a **gate**: `y = σ(W x + b) ⊙ x`. A `Linear` computes one number per
element from the input, the sigmoid turns each into a multiplier between 0 and 1, and each
element is multiplied by its own. It is one route, scaled. The same multiplier, put between
two or more routes that are added, is a **parallel architecture**:

<svg width="720" height="520" viewBox="0 0 720 520" xmlns="http://www.w3.org/2000/svg" font-family="sans-serif" font-size="13">
<rect width="720" height="520" fill="white"/>
<defs><marker id="m1" markerWidth="8" markerHeight="8" refX="7" refY="4" orient="auto"><path d="M0,0 L8,4 L0,8 z" fill="#444"/></marker></defs>
<text x="10" y="20" font-weight="bold" fill="#222">highway (vectormesh Highway)</text>
<text x="10" y="104" fill="#222">x</text>
<text x="4" y="124" font-family="monospace" font-size="10" fill="#222">(B, 64)</text>
<line x1="22" y1="100" x2="48" y2="100" stroke="#444" stroke-width="1.5" marker-end="url(#m1)"/>
<rect x="50" y="88" width="70" height="24" rx="6" fill="#e4e4e4" stroke="#555"/>
<text x="85" y="104" text-anchor="middle" font-size="11" fill="#222">LayerNorm</text>
<line x1="120" y1="100" x2="135" y2="100" stroke="#444" stroke-width="1.5"/>
<path d="M 135 100 L 135 55 L 168 55" fill="none" stroke="#444" stroke-width="1.5" marker-end="url(#m1)"/>
<rect x="170" y="35" width="130" height="40" rx="8" fill="#f6c7a8" stroke="#555"/>
<text x="235" y="52" text-anchor="middle" fill="#222">block T</text>
<text x="235" y="67" text-anchor="middle" font-size="10" fill="#222">Linear, GELU, Linear</text>
<line x1="300" y1="55" x2="361" y2="55" stroke="#444" stroke-width="1.5" marker-end="url(#m1)"/>
<text x="331" y="47" text-anchor="middle" font-family="monospace" font-size="11" fill="#222">(B, 64)</text>
<circle cx="380" cy="55" r="18" fill="#f3d9e6" stroke="#555"/>
<text x="380" y="60" text-anchor="middle" fill="#222">× g</text>
<line x1="135" y1="100" x2="168" y2="100" stroke="#444" stroke-width="1.5" marker-end="url(#m1)"/>
<rect x="170" y="87" width="130" height="26" rx="8" fill="#f3d9e6" stroke="#555"/>
<text x="235" y="104" text-anchor="middle" font-size="11" fill="#222">g = σ(Linear(64, 64))</text>
<line x1="300" y1="100" x2="368" y2="74" stroke="#444" stroke-width="1.2" stroke-dasharray="3,3" marker-end="url(#m1)"/>
<line x1="300" y1="100" x2="366" y2="131" stroke="#444" stroke-width="1.2" stroke-dasharray="3,3" marker-end="url(#m1)"/>
<text x="235" y="127" text-anchor="middle" font-family="monospace" font-size="11" fill="#222">g: (B, 64)</text>
<path d="M 135 100 L 135 150 L 359 150" fill="none" stroke="#444" stroke-width="1.5" marker-end="url(#m1)"/>
<text x="150" y="145" font-size="11" fill="#222">skip: LN(x) itself, (B, 64), no weights</text>
<circle cx="380" cy="150" r="20" fill="#f3d9e6" stroke="#555"/>
<text x="380" y="154" text-anchor="middle" font-size="11" fill="#222">× (1−g)</text>
<path d="M 398 55 L 470 55 L 470 82" fill="none" stroke="#444" stroke-width="1.5" marker-end="url(#m1)"/>
<path d="M 400 150 L 470 150 L 470 118" fill="none" stroke="#444" stroke-width="1.5" marker-end="url(#m1)"/>
<circle cx="470" cy="100" r="16" fill="#dccdf0" stroke="#555"/>
<text x="470" y="106" text-anchor="middle" font-size="18" fill="#222">+</text>
<line x1="486" y1="100" x2="588" y2="100" stroke="#444" stroke-width="1.5" marker-end="url(#m1)"/>
<text x="537" y="92" text-anchor="middle" font-family="monospace" font-size="11" fill="#222">(B, 64)</text>
<text x="596" y="104" fill="#222">y</text>
<text x="10" y="186" font-size="12" fill="#222">y = g ⊙ T(LN(x)) + (1 − g) ⊙ LN(x): one weight per element; the two weights of an element sum to 1</text>
<line x1="10" y1="203" x2="710" y2="203" stroke="#bbb" stroke-width="1"/>
<text x="10" y="224" font-weight="bold" fill="#222">mixture of experts (MoE)</text>
<text x="10" y="304" fill="#222">x</text>
<line x1="24" y1="300" x2="90" y2="300" stroke="#444" stroke-width="1.5"/>
<text x="28" y="292" font-family="monospace" font-size="11" fill="#222">(B, 64)</text>
<line x1="90" y1="250" x2="90" y2="400" stroke="#444" stroke-width="1.5"/>
<line x1="90" y1="250" x2="168" y2="250" stroke="#444" stroke-width="1.5" marker-end="url(#m1)"/>
<line x1="90" y1="300" x2="168" y2="300" stroke="#444" stroke-width="1.5" marker-end="url(#m1)"/>
<line x1="90" y1="350" x2="168" y2="350" stroke="#444" stroke-width="1.5" marker-end="url(#m1)"/>
<line x1="90" y1="400" x2="168" y2="400" stroke="#444" stroke-width="1.5" marker-end="url(#m1)"/>
<rect x="170" y="236" width="130" height="28" rx="8" fill="#f6c7a8" stroke="#555"/>
<text x="235" y="254" text-anchor="middle" font-size="12" fill="#222">expert 1: Linear(64, 64)</text>
<rect x="170" y="286" width="130" height="28" rx="8" fill="#f6c7a8" stroke="#555"/>
<text x="235" y="304" text-anchor="middle" font-size="12" fill="#222">expert 2: Linear(64, 64)</text>
<text x="235" y="331" text-anchor="middle" fill="#222">⋮</text>
<rect x="170" y="336" width="130" height="28" rx="8" fill="#f6c7a8" stroke="#555"/>
<text x="235" y="354" text-anchor="middle" font-size="12" fill="#222">expert N: Linear(64, 64)</text>
<line x1="300" y1="250" x2="361" y2="250" stroke="#444" stroke-width="1.5" marker-end="url(#m1)"/>
<line x1="300" y1="300" x2="361" y2="300" stroke="#444" stroke-width="1.5" marker-end="url(#m1)"/>
<line x1="300" y1="350" x2="361" y2="350" stroke="#444" stroke-width="1.5" marker-end="url(#m1)"/>
<text x="331" y="242" text-anchor="middle" font-family="monospace" font-size="11" fill="#222">(B, 64)</text>
<circle cx="380" cy="250" r="18" fill="#f3d9e6" stroke="#555"/>
<text x="380" y="255" text-anchor="middle" fill="#222">× g₁</text>
<circle cx="380" cy="300" r="18" fill="#f3d9e6" stroke="#555"/>
<text x="380" y="305" text-anchor="middle" fill="#222">× g₂</text>
<circle cx="380" cy="350" r="18" fill="#f3d9e6" stroke="#555"/>
<text x="380" y="354" text-anchor="middle" font-size="11" fill="#222">× g_N</text>
<rect x="170" y="386" width="150" height="28" rx="8" fill="#f3d9e6" stroke="#555"/>
<text x="245" y="404" text-anchor="middle" font-size="11" fill="#222">router: Linear(64, N), softmax</text>
<line x1="320" y1="400" x2="340" y2="400" stroke="#444" stroke-width="1.5"/>
<circle cx="340" cy="400" r="3" fill="#444"/>
<line x1="340" y1="400" x2="340" y2="262" stroke="#444" stroke-width="1.2" stroke-dasharray="3,3"/>
<line x1="340" y1="262" x2="364" y2="259" stroke="#444" stroke-width="1.2" stroke-dasharray="3,3" marker-end="url(#m1)"/>
<line x1="340" y1="312" x2="364" y2="309" stroke="#444" stroke-width="1.2" stroke-dasharray="3,3" marker-end="url(#m1)"/>
<line x1="340" y1="362" x2="364" y2="359" stroke="#444" stroke-width="1.2" stroke-dasharray="3,3" marker-end="url(#m1)"/>
<text x="398" y="278" font-family="monospace" font-size="10" fill="#222">g₁ (B, 1)</text>
<text x="398" y="328" font-family="monospace" font-size="10" fill="#222">g₂ (B, 1)</text>
<text x="398" y="378" font-family="monospace" font-size="10" fill="#222">g_N (B, 1)</text>
<text x="352" y="420" font-family="monospace" font-size="10" fill="#222">g (B, N), split into N columns; the N sum to 1</text>
<path d="M 398 250 L 470 250 L 470 282" fill="none" stroke="#444" stroke-width="1.5" marker-end="url(#m1)"/>
<line x1="398" y1="300" x2="452" y2="300" stroke="#444" stroke-width="1.5" marker-end="url(#m1)"/>
<path d="M 398 350 L 470 350 L 470 318" fill="none" stroke="#444" stroke-width="1.5" marker-end="url(#m1)"/>
<circle cx="470" cy="300" r="16" fill="#dccdf0" stroke="#555"/>
<text x="470" y="306" text-anchor="middle" font-size="18" fill="#222">+</text>
<line x1="486" y1="300" x2="588" y2="300" stroke="#444" stroke-width="1.5" marker-end="url(#m1)"/>
<text x="537" y="292" text-anchor="middle" font-family="monospace" font-size="11" fill="#222">(B, 64)</text>
<text x="596" y="304" fill="#222">y</text>
<text x="10" y="452" font-size="12" fill="#222">y = g₁ · expert₁(x) + … + g_N · expert_N(x), with g = softmax(router(x)): one weight per expert, per window;</text>
<text x="10" y="470" font-size="12" fill="#222">each g_i, (B, 1), multiplies all 64 features of its expert's (B, 64) output (broadcast)</text>
<text x="10" y="496" font-size="12" fill="#222">sparse (k = 1): per window only the expert with the largest g_i runs, its output times g_i; the other routes are not computed</text>
</svg>

**Highway** (Srivastava and colleagues, 2015, the same year as ResNet; lesson 2 drew it in
2.2.3). Two routes leave `x`: through a block `T`, and around it, the skip. A gate computes
`g = σ(Linear(64, 64)(x))`, one multiplier per element, from `x` (lesson 2 wrote this
multiplier `Γ`; here it is `g`). The block's output is
weighted by `g`, the skip by `1 − g`, and the two are added: `y = g ⊙ T(x) + (1 − g) ⊙ x`.
Where `g` is near 1 the block's output passes, where it is near 0 the input passes unchanged.
The skip of lesson 2, `y = T(x) + x`, is the same two routes with no gate: both always pass
in full. vectormesh's `Highway` puts a LayerNorm on `x` first, as its `Skip` does
(lesson 2, 2.2.1.3), so in the drawing and in this notebook
`y = g ⊙ T(LN(x)) + (1 − g) ⊙ LN(x)`, with `g` computed from `LN(x)`.

**Mixture of experts** (MoE; Jacobs and Jordan, 1991). A highway with more routes. Each of
`N` **experts** is a block of its own; here every expert is a `Linear(64, 64)`. The **router**
is a `Linear(64, N)` followed by a softmax: for every window it gives `N` weights
`g_1 … g_N`, each between 0 and 1, together 1, a `(B, N)` tensor. Column `i` of it, `g_i`, is
one number per window, `(B, 1)`; it multiplies all 64 features of expert `i`'s output (the
same number for each feature: broadcasting), and the weighted outputs are added. A **dense**
MoE (vectormesh `MoE`) runs every expert on every window.

**Sparse MoE** (Shazeer and colleagues, 2017; the Switch Transformer, Fedus and colleagues,
2021). The router chooses the `k` experts with the largest weights for each window, and only
those run on it. With `k = 1`, as in most of this notebook, one expert runs per window and its
output is multiplied by its weight `g_i`. vectormesh has it as `SparseMoE`. In large language
models this is how a model can hold thousands of experts and still run only a few per input.

**What must match.** Routes are added element by element, so every route must give the
shape the others give: `T(x)` must have the shape of `x`, and every expert must give the
same `(B, 64)`. The router must give one weight per expert, `(B, N)`. The highway's gate gives
a weight per element, `(B, 64)`; the router a weight per expert and window, `(B, 1)` per
route.

### The models in this notebook

Every model is the same frame with something different in the middle:

`(B, 192)` → `Linear(192, 64)` → GELU → **middle** → `Linear(64, 48)` → `(B, 48)`

The first `Linear` turns four days into 64 features, the last turns 64 features into the
48 half-hours of the forecast. The middle takes `(B, 64)` and gives `(B, 64)`:

| model | middle | it has, besides the frame |
|---|---|---|
| `plain` | nothing | no middle at all: the frame alone |
| `deeper` | `Linear(64, 64)`, GELU | one more plain layer |
| `gate` | `Gate(64)`: `σ(Linear(64, 64)(h)) ⊙ h` | the gate's `Linear(64, 64)` |
| `highway` | `Highway`: LayerNorm, gate `Linear(64, 64)`, block `T` = `Linear(64, 64)`, GELU, `Linear(64, 64)` | the gate, the block and the LayerNorm |
| `moe` | dense `MoE`, 4 experts `Linear(64, 64)`, router `Linear(64, 4)` | 4 experts and a router |
| `sparse_moe` | `SparseMoE`, `N` experts `Linear(64, 64)`, router `Linear(64, N)`, `k = 1` | `N` experts and a router |

`deeper (hidden 86)` and `deeper (hidden 96)` are `deeper` made wider: every layer of the
frame and the middle gets 86 or 96 features instead of 64. They are the controls for the
larger models (1.2). The next cell defines the models and the training. `run` trains one
model and records its curves; it also computes the test MAE of the kept epoch, which nothing
reads until 1.4 reports test once. Two of its arguments are used only in 1.4 and explained
there: `balance`, the weight of a load-balancing loss, and `normalize`, an option of
`SparseMoE` for two experts per window.

In [ ]:
HIDDEN = 64
SEEDS = param((0, 1, 2, 3, 4), test=(0, 1))
BUDGET = param(250, test=3)  # epochs every run trains for (1.3)


def make_model(
    mechanism: str,
    hidden: int = HIDDEN,
    n_experts: int = 4,
    k: int = 1,
    normalize: bool = False,
) -> nn.Module:
    """Linear(192, hidden) -> GELU -> middle -> Linear(hidden, 48), the middle named by
    `mechanism` (see the table above)."""
    if mechanism == "plain":
        return NeuralNet(hidden_size=[W, hidden], out_size=H)
    if mechanism == "deeper":
        return NeuralNet(hidden_size=[W, hidden, hidden], out_size=H)
    inp, act, out = nn.Linear(W, hidden), nn.GELU(), nn.Linear(hidden, H)
    if mechanism == "gate":
        mid = Gate(hidden_size=hidden)
    elif mechanism == "highway":
        # NeuralNet(hidden, hidden) is Linear(hidden, hidden) -> GELU -> Linear(hidden, hidden)
        mid = Highway(transform=NeuralNet(hidden, hidden), hidden_size=hidden)
    elif mechanism == "moe":
        mid = MoE(
            experts=[nn.Linear(hidden, hidden) for _ in range(n_experts)],
            hidden_size=hidden,
            out_size=hidden,
        )
    elif mechanism == "sparse_moe":
        mid = SparseMoE(
            experts=[nn.Linear(hidden, hidden) for _ in range(n_experts)],
            hidden_size=hidden,
            out_size=hidden,
            k=k,
            normalize=normalize,
        )
    else:
        raise ValueError(mechanism)
    return nn.Sequential(inp, act, mid, out)


def n_params(model: nn.Module) -> int:
    """Weights and biases: every number the optimizer changes."""
    return sum(p.numel() for p in model.parameters())


def mae(y: torch.Tensor, yhat: torch.Tensor) -> float:
    return float((y - yhat).abs().mean())


# one batch covering the whole val split; every Trainer below shares this loader
validloader = DataLoader(TensorDataset(xva, yva), batch_size=len(xva))


@dataclass
class Run:
    """One training run: the val MAE and test MAE of the kept epoch, that epoch, the
    curves of every epoch, and the model holding the kept epoch's weights."""

    mae: float
    test_mae: float
    best_epoch: int
    val_mae: list[float]
    val_loss: list[float]
    train_mae: list[float]
    model: nn.Module


def run(
    mechanism: str,
    seed: int,
    epochs: int = BUDGET,
    lr: float = 1e-3,
    batch_size: int = 64,
    balance: float = 0.0,
    **model_kwargs,
) -> Run:
    """Train one model with Adam for `epochs` epochs, record train MAE, val MAE and val loss
    (MSE) after every epoch, and keep the weights of the epoch with the lowest val loss.
    `balance` > 0 adds that multiple of the load-balancing loss of every SparseMoE to the
    training loss (1.4); the val loss never has it."""
    torch.manual_seed(seed)
    model = make_model(mechanism, **model_kwargs)
    trainloader = DataLoader(
        TensorDataset(xtr, ytr), batch_size=batch_size, shuffle=True
    )
    train_curve: list[float] = []
    val_curve: list[float] = []
    val_loss: list[float] = []

    def track(epoch, train_loss, test_loss, metric_dict):
        val_curve.append(metric_dict["MAE"])
        val_loss.append(test_loss)
        with torch.no_grad():
            train_curve.append(mae(ytr, model(xtr)))

    # patience = the whole budget: the Trainer never stops early, and keeps (in memory)
    # the weights of the epoch with the lowest val loss
    settings = TrainerSettings(
        epochs=epochs,
        metrics=[MAE()],
        logdir=Path("modellogs").resolve(),
        train_steps=len(trainloader),
        valid_steps=len(validloader),
        optimizer_kwargs={"lr": lr},
        earlystop_kwargs={"patience": epochs, "save": False, "verbose": False},
    )

    def balanced_step(model: nn.Module, x: torch.Tensor, y: torch.Tensor):
        loss = nn.functional.mse_loss(model(x), y)
        if model.training:
            loss = loss + balance * load_balancing_loss(model)
        return loss

    trainer = Trainer(
        model=model,
        settings=settings,
        loss_fn=nn.MSELoss(),
        optimizer=torch.optim.Adam,
        traindataloader=trainloader,
        validdataloader=validloader,
        step=balanced_step if balance else None,
        reporters=[track],
        progress=False,
    )
    # TrainResult counts epochs from 0; this notebook counts from 1, as the curves do
    result = trainer.loop()
    trainer.model.eval()
    with torch.no_grad():
        test_mae = mae(yte, trainer.model(xte))
    return Run(
        mae=result.metric_dict["MAE"],
        test_mae=test_mae,
        best_epoch=result.epoch + 1,
        val_mae=val_curve,
        val_loss=val_loss,
        train_mae=train_curve,
        model=trainer.model,
    )


def early_stop_epochs(val_loss: list[float], patience: int) -> tuple[int, int]:
    """Where early stopping with `patience` would keep and stop on a recorded val-loss
    curve: the epoch with the lowest loss so far, and the epoch `patience` epochs without
    improvement later (or the last epoch). The same rule as the Trainer's EarlyStopping."""
    best, best_epoch, waited = float("inf"), 0, 0
    for epoch, loss in enumerate(val_loss, start=1):
        if loss < best:
            best, best_epoch, waited = loss, epoch, 0
        else:
            waited += 1
            if waited >= patience:
                return best_epoch, epoch
    return best_epoch, len(val_loss)


def se_diff(a: list[float], b: list[float]) -> float:
    """The difference of two means in standard errors of that difference, as in lessons
    1 and 2: (mean_a - mean_b) / sqrt(std_a**2 / n_a + std_b**2 / n_b)."""
    a, b = np.asarray(a), np.asarray(b)
    se = np.sqrt(a.std(ddof=1) ** 2 / len(a) + b.std(ddof=1) ** 2 / len(b))
    return float((a.mean() - b.mean()) / max(se, 1e-6))


for name in ("plain", "deeper", "gate", "highway", "moe"):
    print(f"{name:8s} (B, 192) -> {tuple(make_model(name)(xva[:8]).shape)}")

## 1.2 The family, each model with its control

A model with more weights can fit more, so a model with a new part that beats a smaller
model may win only because it is larger. The **control** of a claim about a part is the same
model without that part, with as many weights. Each model with a middle has one:

- `gate` against `deeper`: the gate's `Linear(64, 64)` costs exactly what `deeper`'s extra
  layer costs, 64 × 64 + 64 = 4,160 weights;
- `highway` against `deeper (hidden 86)`;
- `moe` and `sparse_moe (N=4)` against `deeper (hidden 96)`;
- `sparse_moe (N=64)` has no control with its total count; 1.5 counts how many of its weights
  one window uses.

`plain` is the **baseline**: the frame without a middle, the score every model must beat.
"Weights" here means every number the optimizer changes, weights and biases: a
`Linear(64, 64)` has 64 × 64 + 64 = 4,160. The next cell counts them, in the frame (the first
and the last `Linear`) and in the middle.

The controls widen every layer, the frame included, so they differ from their arm in where
the extra weights sit as well as in the mechanism. Two arms differ in more than their
mechanism: `highway` is the only model with a LayerNorm, and the experts of `moe` and
`sparse_moe` are single `Linear` layers without a GELU, where `deeper`'s extra layer has one.

In [ ]:
MODEL_CONFIGS = [
    # (label, mechanism, make_model kwargs, control)
    ("plain", "plain", {}, None),
    ("deeper", "deeper", {}, None),
    ("deeper (hidden 86)", "deeper", {"hidden": 86}, None),
    ("deeper (hidden 96)", "deeper", {"hidden": 96}, None),
    ("gate", "gate", {}, "deeper"),
    ("highway", "highway", {}, "deeper (hidden 86)"),
    ("moe", "moe", {"n_experts": 4}, "deeper (hidden 96)"),
    ("sparse_moe (N=4)", "sparse_moe", {"n_experts": 4, "k": 1}, "deeper (hidden 96)"),
    ("sparse_moe (N=64)", "sparse_moe", {"n_experts": 64, "k": 1}, None),
]
CONTROL = {label: control for label, _, _, control in MODEL_CONFIGS}
PARAMS = {
    label: n_params(make_model(mechanism, **kwargs))
    for label, mechanism, kwargs, _ in MODEL_CONFIGS
}

print(
    f"{'model':20s} {'frame':>7s} {'middle':>8s} {'total':>8s}   control (total, difference)"
)
for label, mechanism, kwargs, control in MODEL_CONFIGS:
    h = kwargs.get("hidden", HIDDEN)
    frame = (W * h + h) + (h * H + H)
    against = (
        f"{control} ({PARAMS[control]:,}, {PARAMS[control] - PARAMS[label]:+,})"
        if control
        else ""
    )
    print(
        f"{label:20s} {frame:7,} {PARAMS[label] - frame:8,} {PARAMS[label]:8,}   {against}"
    )

The controls match: `gate` and `deeper` both have 19,632 weights; `highway` has 176
fewer than `deeper (hidden 86)`, and `moe` and `sparse_moe (N=4)` 124 fewer than
`deeper (hidden 96)`. The middle of `highway` holds 12,608: the block `T` (two
`Linear(64, 64)`, 2 × 4,160), the gate's `Linear(64, 64)` (4,160) and the LayerNorm (128, a
scale and a shift for each of the 64 features). The middle of `moe` holds 16,900: four experts
of 4,160 and a router `Linear(64, 4)` of 260. The middle of `sparse_moe (N=64)` holds 270,400:
64 experts and a router `Linear(64, 64)`.

**Everything else is held equal**: the same train, val and test windows, the same seed numbers,
Adam with a learning rate of 1e-3, batches of 64, the MSE as the training loss, and the same
training length and rule for which epoch to keep (1.3). The learning rate is held equal, not
tuned per model: a gated or normalised model might want another step size, so every verdict
below holds at a learning rate of 1e-3 and a batch of 64 (`04_search` is where such settings
get searched). The seed sets the starting weights and the order of the batches; because the
architectures differ, seed 0 of `gate` and seed 0 of `deeper` share only the batch order, not
their starting weights, so the runs of two models are not pairs and are compared unpaired.

## 1.3 How long to train, and which epoch to keep

Train too briefly and a model is still learning; train too long and it fits the train
windows at the cost of the val windows. **Early stopping** (lesson 1,
`1_representation_and_head/03_training`) watches the val loss after every epoch and ends
training once it has not dropped for `patience` epochs in a row, keeping the weights of the
epoch with the lowest val loss. Too small a patience stops a run that bounces up for a few
epochs in the middle of its descent, and a comparison in which some runs are stopped early
compares stopping, not models.

**The check.** Every model trains for a fixed `BUDGET` of 250 epochs, once per seed, and its
train MAE, val MAE and val loss are recorded after every epoch; the weights of the epoch with
the lowest val loss are kept. On each recorded curve the cell then replays early stopping with
a patience of 10 and of 25: which epoch would each have kept, at what val MAE, against the
lowest val loss of the whole budget? Early stopping watches the val *loss* (the MSE), not the
val MAE; the two mostly move together, not always.

**Before you run the next cell**, write down a ranking of the nine models on val, and for each
of `gate`, `highway`, `moe` and `sparse_moe (N=4)` whether you expect it to beat its control.
Also: early stopping ends a run after `patience` epochs without a new lowest val loss. For
a model whose val loss still falls slowly but bounces from epoch to epoch, do you expect
patience 25 to stop it before or after its best epoch?

In [ ]:
runs: dict[str, list[Run]] = {
    label: [run(mechanism, seed, **kwargs) for seed in SEEDS]
    for label, mechanism, kwargs, _ in MODEL_CONFIGS
}

fig, axes = plt.subplots(3, 3, figsize=(13, 9), sharex=True, sharey=True)
for ax, (label, rs) in zip(axes.flat, runs.items()):
    for s, r in zip(SEEDS, rs):
        epochs_axis = range(1, len(r.val_mae) + 1)
        ax.plot(epochs_axis, r.val_mae, color=f"C{s}", lw=0.8, label=f"seed {s}")
        ax.plot(r.best_epoch, r.mae, "o", color=f"C{s}", ms=5)
        keep10, _ = early_stop_epochs(r.val_loss, 10)
        ax.plot(keep10, r.val_mae[keep10 - 1], "x", color=f"C{s}", ms=7, mew=1.5)
    ax.set_title(label, fontsize=10)
    ax.set_ylim(0.135, 0.21)
    ax.grid(alpha=0.3)
for ax in axes[-1]:
    ax.set_xlabel("epoch")
for ax in axes[:, 0]:
    ax.set_ylabel("val MAE")
axes[0, 0].legend(fontsize=7)
fig.suptitle(
    "Val MAE of every run: ● the epoch with the lowest val loss in the budget, "
    "× the epoch patience 10 would keep"
)
plt.tight_layout()
plt.show()


def stopping_summary(runs: dict[str, list[Run]]) -> None:
    """Per model: the kept epochs; the mean val MAE of the epoch kept with patience 10,
    with patience 25 and with the whole budget; the largest loss of one run under
    patience 25 against the budget; and the plateau check: the median val MAE over the
    middle and the last third of the budget (mean over seeds), the drift between them
    (negative: still falling) and how far the kept epoch lies below the last third."""
    third = max(BUDGET // 3, 1)
    print(
        f"{'':30s} {'kept epochs':>25s} {'pat. 10':>7s} {'pat. 25':>7s} {'budget':>7s}"
        f" {'worst 25':>8s} {'middle':>7s} {'last':>7s} {'drift':>7s} {'kept below':>10s}"
    )
    for label, rs in runs.items():
        at = {
            p: [r.val_mae[early_stop_epochs(r.val_loss, p)[0] - 1] for r in rs]
            for p in (10, 25)
        }
        best = [r.mae for r in rs]
        worst = max(a - b for a, b in zip(at[25], best))
        middle = np.mean([np.median(r.val_mae[third : 2 * third]) for r in rs])
        last = np.mean([np.median(r.val_mae[2 * third :]) for r in rs])
        print(
            f"{label:30s} {[r.best_epoch for r in rs]!s:>25s} {np.mean(at[10]):7.4f}"
            f" {np.mean(at[25]):7.4f} {np.mean(best):7.4f} {worst:+8.4f} {middle:7.4f}"
            f" {last:7.4f} {last - middle:+7.4f} {last - np.mean(best):10.4f}"
        )


stopping_summary(runs)

**What the curves show.** Every val curve drops steeply over the first 30 to 60 epochs,
bouncing as it goes. Then the models part ways. The two wider plain stacks and `highway` keep
epochs 47 to 83 and then climb slowly (a positive drift from the middle to the last third):
they start to fit the train windows at the cost of val. `plain` is still drifting down at
epoch 250 (its drift is −0.0038, the only negative one), and keeps epochs 236 to 244 in four of
its five seeds.

**Patience 10 and 25 are too short for some models.** Many × marks lie in the descent. With
patience 25 the worst single run of `plain` keeps an epoch whose val MAE is 0.0110 above the
one it keeps with the whole budget, `moe` 0.0115 and `highway` 0.0124: more than most gaps
between models in the table below, where the seed std of a model is 0.0014 to 0.0028. Under
patience 25 `plain` would score 0.1541 on average, against 0.1501 with the whole budget: a
baseline that is stopped too early moves every comparison made with it. That alone is the
reason for the rule below.

**The rule from here on**: every model trains for the whole budget of 250 epochs, and the
weights of the epoch with the lowest val loss are kept. It keeps the best epoch of every run
(epochs 45 to 244): at the bottom before the climb for the models that climb, and near the
end of the budget for `plain`, which is still falling at epoch 250, so `plain` has no plateau
yet. Its price is a **selection bias**: the lowest of 250 noisy values lies below the typical
level of the epochs around it. The last column does not measure that alone. It is how far the
kept epoch lies below the median of the last third, 0.0078 to 0.0216, and for a model that
climbs after its kept epoch that includes the climb: the largest values belong to `highway`
(0.0216) and the wider plain stacks (0.0185, 0.0178), the models with the largest drift. This
notebook does not separate the bias from the climb. Every row picks its epoch the same way, so
the comparisons below use lesson 2's rule as it is. `plain` might gain a little with a longer
budget; that would only lower the baseline the others are compared with.

## 1.4 Reading the table: one rule, on val

The rule is the one of lessons 1 and 2: a difference between two models counts in
**standard errors of the difference** (`se`), `(mean_a − mean_b) / sqrt(std_a²/n_a + std_b²/n_b)`
over the five seeds. Within about 2 se is no result; 2 to 3 is marginal; beyond 3 counts.
Every conclusion below is drawn on **val**, the split that chooses. A lower MAE is better, so a
negative se means the arm beats the model it is compared with. The spread is over seeds, on
one val year: it says how much a model's score moves with its starting weights and batch
order, not how much it would move on another year.

In [ ]:
val = {label: [r.mae for r in rs] for label, rs in runs.items()}
print(f"{'model':20s} {'val MAE':>16s} {'weights':>9s} {'vs plain':>9s}   vs control")
for label, *_ in MODEL_CONFIGS:
    v = np.asarray(val[label])
    vs_plain = f"{se_diff(v, val['plain']):+.1f} se" if label != "plain" else ""
    control = CONTROL[label]
    vs_control = (
        f"{se_diff(v, val[control]):+.1f} se against {control}" if control else ""
    )
    print(
        f"{label:20s} {v.mean():.4f} ± {v.std(ddof=1):.4f} {PARAMS[label]:9,} "
        f"{vs_plain:>9s}   {vs_control}"
    )

With every model trained for the whole budget, the table shows no gain anywhere.

- **The gate, the case goal 3.1 is about.** `gate` against `plain`: +0.5 se. Against
  `deeper`, with exactly its 19,632 weights: +1.8 se. Both no result: the gate earns nothing,
  and neither does the plain layer that costs as much. A training difference would have
  hidden that. Under patience 25 (1.3, column `pat. 25`) the table would have compared `gate`
  at 0.1511 and `deeper` at 0.1499 with a `plain` stopped too early at 0.1541: both below it,
  where with the whole budget `gate` lies above `plain` (0.1508 against 0.1501). Both sparse
  models would still have been above it (0.1558, 0.1561). The standard errors of those
  patience-25 gaps are not printed here, so this notebook does not say on which side of a line
  they would have fallen. The control (`deeper`) shows which part of a gain the
  mechanism earns; a baseline trained as long as the others shows whether there is a gain
  at all.
- **Weights in a plain stack buy nothing measurable here**: against `plain`, `deeper` is
  −0.9 se, `deeper (hidden 86)` −1.7 and `deeper (hidden 96)` −0.3.
- **`highway`** is +1.5 se against `deeper (hidden 86)` and −0.3 against `plain`: no result.
- **`moe`** is −1.2 se against `deeper (hidden 96)` and −1.3 against `plain`: no result, on the
  side of the mixture this time.
- **`sparse_moe (N=4)`** is +3.9 se against `deeper (hidden 96)` and +3.2 against `plain`:
  worse than both, by this rule a result. The router cell below shows what it computes after
  training.
- **`sparse_moe (N=64)`**, with 285,872 weights, is +3.8 se against `plain`: worse than the
  baseline.

No mechanism beats its control and none beats `plain`; the sparse mixtures are worse than
both. What would have meant the opposite: an arm below its control by more than 2 se.

### The test windows, once

The comparisons are settled on val. The test year took part in no decision: not in the epoch
kept, not in the training length, not in any conclusion above. It is scored here once, with
the weights each run kept, to see whether the val reading holds on a year no choice has seen.

In [ ]:
for label, *_ in MODEL_CONFIGS:
    t = [r.test_mae for r in runs[label]]
    control = CONTROL[label]
    vs_control = (
        f"{se_diff(t, [r.test_mae for r in runs[control]]):+.1f} se against {control}"
        if control
        else ""
    )
    print(
        f"{label:20s} test MAE {np.mean(t):.4f} ± {np.std(t, ddof=1):.4f}   {vs_control}"
    )

On test no mechanism beats its control either: `gate` −0.0 se, `highway` −0.3 and `moe`
+0.5, all no result; `sparse_moe (N=4)` +4.5 se, worse, as on val. Every model has a higher MAE
on test (0.1568 to 0.1638) than on val (0.1476 to 0.1564): partly because the epoch was chosen
on val, partly because the two years differ; this notebook cannot separate the two. The
standard errors of 1.4 are for one val year: another year could move a verdict that lies near
a line.

### Where do the experts' windows go?

A sparse router sends each window to one expert. If it sends nearly all windows to a few
experts, the rest are weights that never see data; if it sends all of them to one, the model
has turned into a single expert and the others are dead weight. That is **router collapse**.
Two things could produce it: the router starts out concentrated (its weights are random), or
training concentrates it.

The cell below takes each seed's trained `sparse_moe` from 1.3, with 4 and with 64 experts,
and counts how many of the 361 val windows its router sends to each expert. It does the same
for the untrained model with the same seed. It also prints the mean **top weight**: the
router's weight `g_i` of the chosen expert, the number that expert's output is multiplied by.

**Predict first**: are more or fewer experts in use after training than before?

In [ ]:
def routing(model: nn.Module) -> tuple[torch.Tensor, float]:
    """For a Linear -> GELU -> SparseMoE -> Linear model: how many val windows the router
    sends to each expert (at k = 2 every window counts for both its experts), and the mean
    weight of the first chosen expert in the output (the router's softmax weight, rescaled
    over the k chosen ones when `normalize` is on)."""
    inp, act, moe = model[0], model[1], model[2]
    with torch.no_grad():
        g = torch.softmax(moe.router(act(inp(xva))), dim=-1)
        top, chosen = g.topk(moe.k, dim=-1)
        if moe.normalize:
            top = top / top.sum(dim=-1, keepdim=True)
    counts = torch.bincount(chosen.reshape(-1), minlength=moe.num_experts)
    return counts, float(top[:, 0].mean())


def describe(counts: torch.Tensor, top_weight: float) -> str:
    top = counts.sort(descending=True).values
    return (
        f"{int((counts > 0).sum()):2d} of {len(counts)} experts used   "
        f"largest expert {int(top[0]):3d} of {len(xva)} windows   "
        f"top weight {top_weight:.3f}"
    )


for n_experts in (4, 64):
    label = f"sparse_moe (N={n_experts})"
    print(label)
    for seed, r in zip(SEEDS, runs[label]):
        torch.manual_seed(seed)
        untrained = make_model("sparse_moe", n_experts=n_experts, k=1)
        print(f"  seed {seed}  untrained: {describe(*routing(untrained))}")
        print(f"          trained:   {describe(*routing(r.model))}")

**The untrained routers are already uneven, and training collapses them.** Untrained, the
router of `sparse_moe (N=4)` uses 3 or 4 of its 4 experts, its largest expert taking 151 to
310 of the 361 windows, and the one of `N=64` uses 9 to 16 of 64, its largest expert taking
109 to 150. So the random start already favours some experts (the first of the two
causes above); training then concentrates the windows further (the second). After training, every seed of `N=4` sends all 361 windows to one
expert, and every seed of `N=64` sends 360 or 361 of them to one (one or two experts used).
The top weight says the same: untrained it is about `1/N` (0.278 to 0.311 with 4 experts,
0.022 to 0.024 with 64); trained it is 0.792 to 0.986, a router nearly certain of its one
expert.

So after training `sparse_moe` is the frame with one `Linear(64, 64)` in the middle, without
a GELU, its output multiplied by the top weight. That weight is still computed by the router
for each window: a per-window scale, the one nonlinear thing the collapsed middle keeps. Two
linear maps in a row (the expert and the last `Linear`) compute one linear map, so the
collapsed model is `plain` with one more linear map and that scale. In 1.4 it scores worse
even than `plain` (+3.2 se), while `deeper` with 19,632 weights and `plain` with 15,472 do
better. So its loss is not explained by the 19,892 weights it uses for a window (1.5 counts
them). Why the collapsed model is worse than `plain` (the routing changing during training,
the learned scale) is not tested here. With 64 experts the other 62 or 63 are weights no val
window reaches after training. Why the router collapses is
not tested here. The usual account is a feedback loop: an expert that receives a few more
windows gets more gradient, becomes better on them, and is chosen more often. The next cell
tests the standard remedy.

### What a balancing loss does

**How the router learns at all.** The router's choice of expert (the largest weight) has no
gradient: a small change of the router's weights does not change which expert is largest. What
has a gradient is the weight `g_i` the chosen expert's output is multiplied by: the loss
changes with `g_i`, `g_i` with the router's weights, so the router learns to give a larger
weight to an expert whose output lowers the loss, and that expert becomes the choice for more
windows.

The Switch Transformer adds an auxiliary loss to the training loss,
`load_balancing_loss`: `N · Σ f_i · P_i`, with `f_i` the fraction of windows sent to expert `i`
and `P_i` the router's mean weight for it over the batch. When every expert gets `1/N` of the
windows and `1/N` of the weight, the sum is `N · N · (1/N · 1/N) = 1`; when one expert gets
every window and all the weight, it is `N · 1 · 1 = N`. Only `P_i` has a gradient (`f_i` is a
count), so the loss pushes the router's *mean* weights towards `1/N` for every expert. It is
added with a factor, the balancing weight, and only while training: the val loss that picks
the kept epoch is the same MSE as before.

**How strong is a weight?** Fedus and colleagues use 1e-2 next to a cross-entropy of a few
units, where the balancing term is a small fraction of the training loss. Here the training
loss is an MSE of about 0.04, so the same weight is relatively far stronger. The cell prints,
for each arm at its kept epoch and on the whole train split, the MSE, the balancing loss and
the balancing term (weight × loss) as a share of the MSE.

**Arms.** `sparse_moe (N=64)` again, with balancing weights 1e-4, 1e-3 and 1e-2.

**A confound, and the arms that test it.** With `k = 1` the chosen expert's output is
multiplied by its weight `g_i`. A balancing loss pushes the router's mean weights towards
`1/64`, so it may shrink the top weight, and with it the signal through the middle, as well
as spread the windows. If balancing costs accuracy, either could be the cause. So three arms
more, at `k = 2` with `normalize=True`: without balancing, with 3e-4, and with 1e-2. The weight 3e-4 comes from a trial run of one seed
for 25 epochs that tried 0 and 3e-4 and kept 3e-4 because its balancing term came out at about
1% of the MSE; it was chosen on the size of the term alone, not on a val score. The table
below prints the term again (`term/MSE`).
`normalize=True` rescales the two chosen weights so they add up to 1: however flat the
router's weights become, the two experts' outputs are blended with weights that sum to 1, and
the signal is not shrunk. (With `k = 1` it would always give the weight 1, the router would get
no gradient at all, so vectormesh allows it only from `k = 2`.) At `k = 2` every window is
counted for both of its experts, so "experts used" counts the experts that are first or second
choice of some window: it is not on the same footing as the `k = 1` count.

**Predict first**: with a loss that pushes the usage towards many experts, what happens to
the val MAE, at `k = 1` and at `k = 2` normalized?

In [ ]:
BALANCES = (1e-4, 1e-3, 1e-2)
K2_BALANCES = (0.0, 3e-4, 1e-2)
N_ROUTED = 64

balanced: dict[str, list[Run]] = {"k=1, no balance": runs["sparse_moe (N=64)"]}
for b in BALANCES:
    balanced[f"k=1, balance {b:g}"] = [
        run("sparse_moe", seed, balance=b, n_experts=N_ROUTED, k=1) for seed in SEEDS
    ]
for b in K2_BALANCES:
    balanced[f"k=2 normalized, balance {b:g}"] = [
        run("sparse_moe", seed, balance=b, n_experts=N_ROUTED, k=2, normalize=True)
        for seed in SEEDS
    ]


def loss_terms(model: nn.Module, weight: float) -> tuple[float, float, float]:
    """On the whole train split: the MSE, the balancing loss, and weight x balancing loss as
    a share of the MSE."""
    model.eval()
    with torch.no_grad():
        mse = float(nn.functional.mse_loss(model(xtr), ytr))
        aux = float(load_balancing_loss(model))
    return mse, aux, weight * aux / mse


reference = [r.mae for r in balanced["k=1, no balance"]]
k2_reference = [r.mae for r in balanced["k=2 normalized, balance 0"]]
print(
    f"{'sparse_moe (N=64)':30s} {'val MAE':>16s} {'vs k=1, none':>13s} {'vs k=2, none':>13s}"
    f" {'top weight':>10s} {'train MSE':>9s} {'bal. loss':>9s} {'term/MSE':>9s}   experts used"
)
for label, rs in balanced.items():
    weight = float(label.split("balance ")[1]) if "balance " in label else 0.0
    v = np.array([r.mae for r in rs])
    used, weights = zip(*[routing(r.model) for r in rs])
    mse, aux, share = map(np.mean, zip(*[loss_terms(r.model, weight) for r in rs]))
    vs_k2 = f"{se_diff(v, k2_reference):+10.1f} se" if label.startswith("k=2") else ""
    print(
        f"{label:30s} {v.mean():.4f} ± {v.std(ddof=1):.4f} {se_diff(v, reference):+10.1f} se"
        f" {vs_k2:>13s} {np.mean(weights):10.3f} {mse:9.4f} {aux:9.2f} {share:9.1%}"
        f"   {[int((c > 0).sum()) for c in used]}"
    )
print()
stopping_summary({k: v for k, v in balanced.items() if k != "k=1, no balance"})

**The size of the dose.** Next to a train MSE of 0.0358 to 0.0467, the balancing term at
`k = 1` is 7.1% of the MSE at 1e-4, 9.4% at 1e-3 and 27.2% at 1e-2; at `k = 2` it is 1.1% at
3e-4 and 23.8% at 1e-2. The weight of the paper, 1e-2, is here about a quarter of the training
loss.

**At `k = 1` a strong dose spreads the windows and costs accuracy.** 1e-4 changes little
(+0.4 se). 1e-3 spreads the windows over 2 to 9 experts and 1e-2 over 22 to 33, and the val MAE
rises from 0.1564 to 0.1631 (+4.1 se) and 0.1738 (+8.9 se). The top weight falls with it:
0.859, then 0.172, then 0.060. At 1e-2 the chosen expert's output reaches the last `Linear`
multiplied by about 0.06, so the `k = 1` arms change two things at once.

**At `k = 2` normalized** the two chosen weights always sum to 1 (the top one is 0.722, 0.659
and 0.589 on average). Without balancing the val MAE is 0.1539, level with the collapsed
`k = 1` model (−1.7 se), with 4 to 10 experts used. At 3e-4, a term of 1.1% of the MSE, the
windows spread over 15 to 25 experts and the val MAE is 0.1516, −1.5 se against no balancing:
no result. The difference is 0.1516 − 0.1539 = −0.0023 at 1.5 se, so one se is about 0.0015,
and the comparison rules out a cost above about 0.003 MAE (2 se); a smaller cost it cannot
see. The same arm is −2.5 se against the collapsed `k = 1` model: marginal, and the lowest
val MAE of the sparse arms. At 1e-2, a term of 23.8%, the windows spread over 36 to 43 experts
and the val MAE is 0.1602, +3.4 se. The two dosed arms differ both in the size of the term
(1.1% against 23.8%) and in the spread (15 to 25 against 36 to 43 experts), so this table does
not say which of the two cost accuracy: a moderate spread at a term of 1% cost nothing
measurable, a wider spread at a term of a quarter of the MSE cost +3.4 se. (At `k = 2` "experts used" counts
first and second choices, so it is not on the same footing as the `k = 1` counts.)

The balanced arms kept epochs 32 to 243; at `k = 1` the arms at 1e-4 and 1e-2 still drift down
a little in the last third (−0.0012 and −0.0034).

A balanced router is one that satisfies the balancing loss; whether it forecasts better
depends on the dose and on the data. Here a light dose spread the windows with no cost this
table can see and no gain over `k = 2` without balancing. Usage is a diagnosis (is the router collapsed?), not a goal.

## 1.5 Weights trained, weights used, and latency

A sparse MoE trains every expert, but runs only the chosen ones on a window. So it has two
counts: the **total** weights, all that are trained and stored, and the **active** (used)
weights per window, the ones that take part in computing one window's forecast: the frame,
the router, and the `k` chosen experts. A dense MoE runs every expert, so for it the two are
the same.

In [ ]:
def active_weights(label: str, n_experts: int, k: int) -> int:
    """Weights one window passes through: the total minus the experts it is not routed to."""
    per_expert = HIDDEN * HIDDEN + HIDDEN  # Linear(64, 64)
    return PARAMS[label] - (n_experts - k) * per_expert


for label, n_experts, k in [
    ("moe", 4, 4),
    ("sparse_moe (N=4)", 4, 1),
    ("sparse_moe (N=64)", 64, 1),
]:
    active = active_weights(label, n_experts, k)
    print(
        f"{label:18s} total {PARAMS[label]:8,}   active per window {active:7,}   "
        f"total / active {PARAMS[label] / active:5.1f}"
    )

`sparse_moe (N=64)` stores 285,872 weights and uses 23,792 for one window: the frame
(15,472), the router `Linear(64, 64)` (4,160) and one expert (4,160), 12.0 times fewer than it
has. `sparse_moe (N=4)` uses 19,892 of its 32,372; dense `moe` uses all 32,372. The router is a
`Linear(64, N)`, so the active count grows with `N` even at one expert per window. After
training, the `N=64` router of 1.4 sends (nearly) every val window to one expert: the other
experts are stored and counted, but no val window passes through them.

**Does a model that uses fewer weights per window run faster?** Speed here is **latency**:
the time one forward pass takes, from a batch in to a forecast out. Look at what the code
does. Dense `MoE` calls every expert once on the whole batch, then stacks the outputs and adds
them weighted. `SparseMoE` sorts the windows by their chosen expert, and then, for every
expert that received at least one window, takes out those windows (a *gather*), runs the
expert on them, multiplies by the weights and adds the result back into place with
`index_add_` (a *scatter*). Both loop over the experts in Python, one call after another.

A CPU runs PyTorch operations on several **threads** (cores working on one operation at
once); `torch.get_num_threads()` says how many. Splitting a large operation over threads
saves time; for a tiny one, waking the threads can cost more than the work. So the same
code can be timed at the default number of threads and at one.

**Question.** Does the time of a forward pass follow the weights used per window, or
something else?

**Setup.** First, what one visit to an expert costs: each operation `SparseMoE` runs per
expert, timed alone on a few windows, at the default number of threads and at one. Then the
three arms at `k = 1`, a batch of 64 of this notebook's windows, and `N` from 4 to 256
experts: dense `MoE`; `SparseMoE` with its own freshly made router; and `SparseMoE` with an
even router that sends window `i` to expert `i mod N`, the spread a balancing loss aims for.
Both thread counts again. Every arm runs without gradients (`torch.no_grad()`, as a forecast in
use does), each time is the median of several repeats, and the input is seeded.

**Predict first**: which arm grows slowest with `N`, and is sparse faster or slower than
dense at `N = 256`? Does your answer depend on the number of threads?

In [ ]:
N_EXPERTS_TO_CHECK = param((4, 16, 64, 256), test=(2, 4))
ROUTING_BATCH = 64
CALLS = param(30, test=3)  # forward passes per timing
REPEATS = param(7, test=2)  # timings per arm; the median is reported
DEFAULT_THREADS = torch.get_num_threads()
THREADS = (DEFAULT_THREADS, 1)


def time_call(f) -> float:
    """Median over REPEATS of the mean time of CALLS calls of f(), in ms."""
    times = []
    with torch.no_grad():
        for _ in range(5):
            f()
        for _ in range(REPEATS):
            start = time.perf_counter()
            for _ in range(CALLS):
                f()
            times.append((time.perf_counter() - start) / CALLS * 1000)
    return float(np.median(times))


class EvenRouter(nn.Module):
    """A router that spreads a batch over the experts perfectly evenly (window i goes to
    expert i mod N), whatever the input."""

    def __init__(self, n_experts: int):
        super().__init__()
        self.n_experts = n_experts

    def forward(self, h: torch.Tensor) -> torch.Tensor:
        chosen = torch.arange(len(h)) % self.n_experts
        return 10.0 * nn.functional.one_hot(chosen, self.n_experts).float()


def experts(n: int) -> list[nn.Module]:
    return [nn.Linear(HIDDEN, HIDDEN) for _ in range(n)]


# a seeded batch of this notebook's windows, projected to HIDDEN once: the block under
# test is the mixture of experts, not the input projection
torch.manual_seed(0)
with torch.no_grad():
    probe_x = nn.Linear(W, HIDDEN)(xtr[:ROUTING_BATCH])

# one expert visit of SparseMoE on 4 windows, operation by operation
one = nn.Linear(HIDDEN, HIDDEN)
rows = torch.arange(0, ROUTING_BATCH, 16)  # 4 windows
weight = torch.rand(len(rows), 1)
out = torch.zeros(ROUTING_BATCH, HIDDEN)
ops = {
    "gather x[rows]": lambda: probe_x[rows],
    "expert Linear(64, 64)": lambda: one(probe_x[rows]),
    "multiply by g_i": lambda: probe_x[rows] * weight,
    "scatter index_add_": lambda: out.index_add_(0, rows, probe_x[rows]),
}
print(f"default threads: {DEFAULT_THREADS}")
print(
    f"{'one expert visit, 4 windows':28s}"
    + "".join(f"{t:>3d} thread(s) " for t in THREADS)
)
for name, f in ops.items():
    cells = []
    for threads in THREADS:
        torch.set_num_threads(threads)
        cells.append(f"{time_call(f) * 1000:9.1f} µs ")
    print(f"{name:28s}" + "".join(cells))
torch.set_num_threads(DEFAULT_THREADS)

timings = {t: {"dense": {}, "sparse": {}, "sparse, even": {}} for t in THREADS}
reached: dict[int, int] = {}
for threads in THREADS:
    torch.set_num_threads(threads)
    print(f"\n{threads} thread(s)")
    for n in N_EXPERTS_TO_CHECK:
        torch.manual_seed(0)
        sparse = SparseMoE(experts(n), hidden_size=HIDDEN, out_size=HIDDEN, k=1)
        torch.manual_seed(0)
        dense = MoE(experts(n), hidden_size=HIDDEN, out_size=HIDDEN)
        torch.manual_seed(0)
        even = SparseMoE(experts(n), hidden_size=HIDDEN, out_size=HIDDEN, k=1)
        even.router = EvenRouter(n)
        with torch.no_grad():
            reached[n] = int(
                torch.bincount(sparse.router(probe_x).argmax(-1)).count_nonzero()
            )
        t = timings[threads]
        for name, model in [
            ("dense", dense),
            ("sparse", sparse),
            ("sparse, even", even),
        ]:
            t[name][n] = time_call(partial(model, probe_x))
        print(
            f"  N={n:4d}  dense {t['dense'][n]:6.3f} ms   "
            f"sparse {t['sparse'][n]:6.3f} ms ({reached[n]:2d} experts reached)   "
            f"sparse, even {t['sparse, even'][n]:6.3f} ms ({min(n, ROUTING_BATCH):2d} reached)   "
            f"sparse / dense {t['sparse'][n] / t['dense'][n]:4.1f}x   "
            f"even / dense {t['sparse, even'][n] / t['dense'][n]:4.1f}x"
        )
    first, last = min(N_EXPERTS_TO_CHECK), max(N_EXPERTS_TO_CHECK)
    print(
        f"  from N={first} to N={last}: dense grows {t['dense'][last] / t['dense'][first]:.1f}x, "
        f"sparse {t['sparse'][last] / t['sparse'][first]:.1f}x, "
        f"sparse, even {t['sparse, even'][last] / t['sparse, even'][first]:.1f}x"
    )
torch.set_num_threads(DEFAULT_THREADS)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, threads in zip(axes, THREADS):
    for name, marker in [("dense", "o"), ("sparse", "s"), ("sparse, even", "^")]:
        ns = list(timings[threads][name])
        ax.plot(ns, [timings[threads][name][n] for n in ns], marker=marker, label=name)
    ax.set_xscale("log", base=2)
    ax.set_yscale("log")
    ax.set_xlabel("number of experts N (log)")
    ax.set_title(f"{threads} thread(s): forward pass of one batch of 64")
    ax.grid(alpha=0.3, which="both")
axes[0].set_ylabel("time per forward pass (ms, log)")
axes[0].legend()
plt.tight_layout()
plt.show()

Measured on a CPU with 6 threads by default, while other jobs ran on the same machine.
Absolute times vary with the machine and with what else runs on it; read the shape. The
6-thread numbers are this machine under that load: on another machine, or an idle one, a
scatter at 6 threads can cost a different amount, and the size of the gap with it.

- **One expert visit is cheap, except the scatter at 6 threads.** The gather, the expert's
  `Linear(64, 64)` and the multiplication cost the same at 6 threads and at 1 (0.9, 3.1 and
  1.4 µs). `index_add_` costs 216.1 µs at 6 threads and 2.1 µs at 1: on a tensor this small,
  starting six threads costs far more than the work. `SparseMoE` pays it once for every
  expert that receives a window.
- **At 6 threads sparse is slower than dense at every `N`**: 2.0x to 10.8x the time of dense
  with its own router, 6.3x to 20.9x with an even load.
- **At 1 thread the ranking flips from `N = 64` on**: sparse takes 0.5x the time of dense at
  `N = 64` and 0.2x at `N = 256`, with an even load 0.4x at `N = 256`. Dense runs every expert
  on every window and grows 22.5x from `N = 4` to `N = 256`, for 64 times the weights.
- **At both thread counts sparse follows the experts it reaches, not `N`.** With its own
  router it reaches 9 experts at `N = 16` and at `N = 64`, and takes about the same time
  (1.818 and 1.841 ms at 6 threads; 0.117 and 0.142 ms at 1). With an even load it reaches 4,
  16, 64 and 64 experts: its time rises up to `N = 64` and then moves little (8.651 to
  8.757 ms at 6 threads; 0.442 to 0.535 ms at 1). From `N = 16` on, the even load, the spread a
  balancing loss asks for, is the slowest sparse arm at both thread counts; at `N = 4` both
  sparse arms reach the same 4 experts and take about the same time (0.915 and 0.988 ms at 6
  threads; 0.086 and 0.088 ms at 1).

So the weights used per window do not set the latency. Every sparse arm runs one expert per
window (and a router that grows with `N`); its time follows how many experts receive a window
and what each visit costs in this implementation on this machine. Whether fewer active
weights are faster depends on how they are computed: at `N = 256` one setting of the runtime
moves sparse from 2.0x the time of dense to 0.2x. Measure on the machine and with the settings
the model will run with.

## What you now know

- **The control decides an architecture claim.** A model with a new part that beats a
  smaller baseline may only be larger. The control is the same model without the part, with
  as many weights, trained on the same data, with the same learning rate, batch size and
  training length. Here `gate` was no better than `deeper`, which has exactly its
  weights (+1.8 se), and neither was better than `plain` once `plain` trained as long as it
  kept improving (+0.5 and −0.9 se). With patience 25 `plain` was stopped early (0.1541 against
  0.1501 with the whole budget), and `gate` (0.1511) and `deeper` (0.1499) would have lain
  below it; both sparse models (0.1558, 0.1561) would not.
- **A gate is part of a parallel architecture.** A highway weighs a block with `g` and the
  skip around it with `1 − g` and adds them; a mixture of experts has `N` routes and a router,
  `Linear(64, N)` and a softmax, that gives every expert one weight per window. The routes are
  added, so every route gives the same shape, `(B, 64)` here, and the router gives `(B, N)`,
  one `(B, 1)` column per route.
- **Total weights and active weights are different numbers.** `sparse_moe (N=64)` stores 285,872 weights and uses 23,792 for one
  window: the frame, the router (which grows with `N`) and one expert.
- **Router collapse and balancing.** After training, every seed of `sparse_moe` sent all, or all but
  one, of the 361 val windows to one expert; with 4 experts that model was worse than its
  control (+3.9 se) and than `plain` (+3.2 se), and why is not tested here. A balancing loss
  spreads the windows: at a term of 1.1% of the MSE (`k = 2`) it showed no cost above about
  0.003 MAE, at about a quarter of the MSE it cost accuracy.
- **Fewer active weights is not automatically lower latency.** The time of a `SparseMoE` forward pass follows the number of
  experts that receive a window and what each visit costs. Here, at 6 threads on a loaded machine,
  one scatter per expert made sparse slower than dense at every `N`; at 1 thread sparse is faster from `N = 64` on.
- **Read the curves of every run before you trust a stopping rule.** Patience 10 and 25 stopped some runs in their descent, `plain`
  among them (0.1541 under patience 25 against 0.1501). Training every run for the whole
  budget and keeping its lowest val loss kept the best epoch of every run, at the price of a
  selection bias that every row shares: the lowest of many noisy epochs lies below the level
  around it.

`04_search` is where you stop checking hand-picked configurations one at a time and let a
search cover the space, including the learning rate held fixed here.

## Exercises

**E1: predict, then measure.** Before re-running this notebook, write down a ranking of the
nine models on val, and whether you expect `gate` to beat `deeper`. Change `SEEDS` to ten
seeds and check: does any reading of 1.4 change sides of the 2 se line?

**E2: force the gate open.** Initialise the bias of the `Gate`'s `Linear` to a large positive
number (say 10) so the sigmoid gives about 1 for every element, and keep that layer fixed
(`requires_grad_(False)`). The gate then passes its input almost unchanged, and the model
computes what `plain` computes, with 4,160 weights that do nothing. After training, print the
smallest gate value on the val windows to check that it is still about 1 (the first `Linear`
can grow its features during training). Predict, then check: does its val MAE come out level
with `plain`'s (within 2 se)?

**E3: more experts, dense.** Train `make_model("moe")` with 2, 4, 8 and 16 experts, with the
same training as 1.3, and give each a `deeper` control of the same total (find the hidden
width with `n_params`). Does the dense mixture beat its control at any `N`, or does its score
follow its number of weights?

**E4: another state.** `australian_electricity.py` loads series `T1`; the set has five states,
`T1` to `T5`. Set `australian_electricity.ITEM_ID = "T3"` before `load_ausalia()`, re-run the
notebook and compare: does any mechanism beat its control on another state's demand?

**E5: `k = 2` in the timing.** Re-run the timing sweep with `k = 2`, at both thread counts.
Predict first: does sparse time come closer to dense or move further away?

**E6: experts with a GELU.** The experts here are single `Linear` layers, one GELU fewer than
`deeper`'s extra layer. Make every expert `NeuralNet(64, 64)` (a `Linear`, a GELU, a
`Linear`), build a control with as many weights, and compare. Does the gap to the control
change?